In [ ]:

import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


In [ ]:
SR = 32000
DURATION = 5  # seconds
IMG_SIZE = 224
N_MELS = 128
BATCH_SIZE = 16
EPOCHS = 5

In [ ]:
def audio_to_mel(path):
    y, sr = librosa.load(path, sr=SR)
    
    # pad or trim
    target_len = SR * DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel)
    
    # normalize
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
    
    return log_mel

In [ ]:
def split_audio(path):
    y, sr = librosa.load(path, sr=SR)
    chunk_size = SR * DURATION
    
    chunks = []
    for i in range(0, len(y), chunk_size):
        chunk = y[i:i+chunk_size]
        
        if len(chunk) < chunk_size:
            chunk = np.pad(chunk, (0, chunk_size - len(chunk)))
        
        chunks.append(chunk)
    
    return chunks

In [ ]:
df = pd.read_csv("birdclef-2026/train.csv")

labels = sorted(df['primary_label'].unique())
label_to_idx = {l: i for i, l in enumerate(labels)}
num_classes = len(labels)

In [ ]:
num_classes

In [ ]:
def generator(df):
    for _, row in df.iterrows():
        path = f"birdclef-2026/train_audio/{row['filename']}"
        label = label_to_idx[row['primary_label']]
        
        chunks = split_audio(path)
        
        for chunk in chunks:
            mel = librosa.feature.melspectrogram(y=chunk, sr=SR, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel)
            
            log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
            
            img = tf.image.resize(log_mel[..., None], (IMG_SIZE, IMG_SIZE))
            
            y_vec = np.zeros(num_classes)
            y_vec[label] = 1
            
            yield img, y_vec

In [ ]:
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(num_classes,), dtype=tf.float32),
)

dataset = tf.data.Dataset.from_generator(
    lambda: generator(df),
    output_signature=output_signature
)

dataset = dataset.shuffle(512).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models

def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    
    return models.Model(inputs, outputs)

model = build_model()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
model.fit(dataset, epochs=EPOCHS)